In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from src.entropy_metrics import calc_shannon
from src.drift_detection import get_rolling_entropy, calc_delta_h
from src.risk_signal import generate_risk_score, format_risk_message

# Carichiamo la baseline salvata nel Notebook 1
with open('results/logs/baseline_metrics.json', 'r') as f:
    baseline = json.load(f)
print(f"baseline {baseline}")

# Carichiamo il dataset con il drift
df = pd.read_csv('data/drift_dataset.csv')
signal = df['sensor_value'].values

In [ ]:
WINDOW_SIZE = 500
STEP = 100

# Calcoliamo l'entropia reale nel tempo
h_t = get_rolling_entropy(signal, WINDOW_SIZE, STEP, calc_shannon)

# Calcoliamo delta H(t) e i livelli di rischio (Z-score)
delta_h_list = [calc_delta_h(h, baseline['shannon_mean']) for h in h_t]
risk_levels = [generate_risk_score(dh, baseline['shannon_std']) for dh in delta_h_list]

print(f"Analisi completata. Rilevati {len(risk_levels)} segmenti.")

In [ ]:
plt.figure(figsize=(12, 8))

# Plot 1: Entropia vs Baseline
plt.subplot(2, 1, 1)
plt.plot(h_t, color='blue', label='H(t) corrente')
plt.axhline(baseline['shannon_mean'], color='green', linestyle='--', label='Baseline')
plt.title("Rilevazione Drift tramite Variazione Entropica")
plt.legend()

# Plot 2: Segnale di Rischio
plt.subplot(2, 1, 2)
plt.plot(delta_h_list, color='orange', label='ΔH(t)')
plt.axhline(baseline['shannon_std'] * 2, color='orange', linestyle=':', label='Soglia Warning (2σ)')
plt.axhline(baseline['shannon_std'] * 3, color='red', linestyle=':', label='Soglia Critical (3σ)')
plt.fill_between(range(len(risk_levels)), 0, np.array(delta_h_list), 
                 where=np.array(risk_levels)==2, color='red', alpha=0.3)
plt.title("Generazione Segnale di Rischio")
plt.xlabel("Window Index")
plt.legend()

plt.tight_layout()
plt.savefig('results/figures/drift_analysis.png')
plt.show()